# 核心概念
## 指标
### Nvidia 指标
答案准确率
答案准确率衡量的是模型对给定问题的回答与参考标准之间的一致性。该指标通过两个不同的“LLM 评委”提示来实现，每个提示会返回一个评分（0、2 或 4）。该指标会将这些评分转换为 [0,1] 的范围内，然后取评委给出的两个评分的平均值。分数越高，表示模型的答案与参考标准越接近。

- 0 →答复不准确或未解决与参考文献相同的问题。
- 2 →响应部分与参考一致。
- 4 →响应与参考完全一致。



In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy

sample = SingleTurnSample(
    user_input="When was Einstein born?",
    response="Albert Einstein was born in 1879.",
    reference="Albert Einstein was born in 1879."
)

scorer = AnswerAccuracy(llm=evaluator_llm) # evaluator_llm wrapped with ragas LLM Wrapper
score = await scorer.single_turn_ascore(sample)
print(score)

输出

1.0

如何计算
- 步骤 1： LLM 使用两个不同的模板生成评级以确保稳健性：
    - 模板 1： LLM 将回复与参考进行比较，并按0、2 或 4的等级进行评分。
    - 模板 2： LLM 再次评估相同的问题，但这次答案和参考文献的角色互换了。
   
  这种双视角方法保证了对答案准确性的公平评估。

- 步骤2：如果两个评分均有效，则最终得分为得分1和得分2的平均值；否则，取有效得分。

计算示例：

- 用户输入： “爱因斯坦何时出生？”
- 回答： “阿尔伯特·爱因斯坦出生于 1879 年。”
- 参考： “阿尔伯特·爱因斯坦出生于 1879 年。”

假设两个模板都返回4分（表示完全匹配），则转换如下：

- 评级4对应于[0,1] 范围内的1 。
- 计算两个分数的平均值： (1 + 1) / 2 = 1。

因此，最终的答案准确率得分为1。

### 相似拉格斯指标
1. 答案正确性：该指标通过考虑语义和事实的相似性来衡量生成的答案与基本事实相比的准确性。

2. 评分细则：基于评分细则的标准评分标准允许基于用户自定义的评分细则进行评估，每个评分细则都概述了具体的评分标准。法学硕士 (LLM) 会根据这些自定义描述来评估答案，确保评估过程的一致性和客观性。

#### 指标比较
答案正确性与答案准确性
- LLM 调用：答案正确性需要三次 LLM 调用（两次用于将响应和参考分解为独立语句，一次用于对它们进行分类），而答案准确性使用两个独立的 LLM 判断。
- 令牌使用：由于答案正确性的详细细分和分类过程，它会消耗更多的令牌。
- 可解释性：答案正确性通过提供对事实正确性和语义相似性的详细见解提供了高度的可解释性，而答案准确度则提供了直接的原始分数。
- 稳健评估：答案准确性通过双重 LLM 评估确保一致性，而答案正确性通过深入评估响应的质量提供整体视图。

答案准确度与评分标准分数
- LLM 评分：答案准确度需要进行两次评分（每个 LLM 评委一次），而评分标准分数只需要一次。
- 令牌使用：答案准确度很小，因为它只输出分数，而评分标准分数会产生推理，从而增加令牌消耗。
- 可解释性：答案准确度提供的是原始分数，没有任何依据，而评分标准分数则提供了带有结论的推理。
- 效率：答案准确度很轻量，并且适用于较小的模型。




### 语境相关性
上下文相关性评估检索到的上下文（块或段落）是否与用户输入相关。此操作通过两次独立的“LLM-as-a-judge”提示调用完成，每次调用都会以0、1 或 2的等级对相关性进行评分。然后将评分转换为 [0,1] 的等级，并取平均值得出最终分数。分数越高，表示上下文与用户查询的匹配程度越高。

- 0 → 检索到的上下文与用户的查询完全不相关。
- 1 → 上下文部分相关。
- 2 → 上下文完全相关。

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import ContextRelevance

sample = SingleTurnSample(
    user_input="When and Where Albert Einstein was born?",
    retrieved_contexts=[
        "Albert Einstein was born March 14, 1879.",
        "Albert Einstein was born at Ulm, in Württemberg, Germany.",
    ]
)

scorer = ContextRelevance(llm=evaluator_llm)
score = await scorer.single_turn_ascore(sample)
print(score)

输出

1.0


如何计算

步骤 1： LLM 需要使用两个不同的模板（template_relevance1 和 template_relevance2）来评估检索到的上下文与用户查询的相关性。每个模板都会返回0、1或2的相关性评分。

步骤 2：将每个评分除以 2，使其标准化为 [0,1] 范围内的数值。如果两个评分均有效，则最终得分为这些标准化值的平均值；如果只有一个有效，则使用该得分。

计算示例：

- 用户输入： “阿尔伯特·爱因斯坦何时何地出生？”
- 检索上下文：
- “阿尔伯特·爱因斯坦出生于 1879 年 3 月 14 日。”
- “阿尔伯特·爱因斯坦出生于德国符腾堡州乌尔姆。”

在此示例中，两个检索到的上下文通过提供阿尔伯特·爱因斯坦的出生日期和地点，完全满足用户的查询要求。因此，两个提示都会将合并后的上下文评分为2（完全相关）。将每个分数标准化后，结果为1.0（2/2），将两个结果取平均值，最终上下文相关性得分仍为1。

#### 相似拉格斯指标
- 上下文准确率：它衡量检索到的上下文中与回答用户查询相关的部分所占的比例。它以所有检索到的语块的平均准确率@k 计算，表明检索系统对相关信息进行排序的准确率。

- 上下文召回率：它量化了相关信息被成功检索的程度。其计算方法是，检索结果中发现的相关主张（或上下文）数量与参考文献中相关主张总数的比率，以确保不遗漏重要信息。

- 评分标准：基于评分标准的评分指标根据用户自定义的评分标准对答案进行评估，确保评估结果的一致性和客观性。评分标准灵活，可满足用户需求。


上下文准确率、上下文回忆率与上下文相关性
- LLM 调用：上下文精度和上下文召回率各需要一次 LLM 调用，一次验证上下文有用性以获取参考（判决“1”或“0”），一次将每个答案句子分类为可归因（二进制“是”（1）或“否”（0）），而上下文相关性使用两次 LLM 调用以增强稳健性。
- 令牌使用情况：上下文准确率和上下文召回率消耗更多令牌，而上下文相关性则更高效地使用令牌。
- 可解释性：上下文准确率和上下文召回率通过详细推理提供高度可解释性，而上下文相关性则提供没有解释的原始分数。
- 稳健评估：与上下文准确率和上下文召回率的单次调用方法相比，上下文相关性通过双重 LLM 判断提供更稳健的评估。

### 响应的基础性
响应的基础性衡量的是检索到的语境对回应的支持程度或“立足点”。它评估回应中的每个主张是否可以在提供的语境中找到全部或部分依据。

- 0 → 该回应根本没有基于上下文。
- 1 → 响应部分接地。
- 2 → 响应完全有根据（每个语句都可以从检索到的上下文中找到或推断出来）。



In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import ResponseGroundedness

sample = SingleTurnSample(
    response="Albert Einstein was born in 1879.",
    retrieved_contexts=[
        "Albert Einstein was born March 14, 1879.",
        "Albert Einstein was born at Ulm, in Württemberg, Germany.",
    ]
)

scorer = ResponseGroundedness(llm=evaluator_llm)
score = await scorer.single_turn_ascore(sample)
print(score)

如何计算
- 步骤1：法学硕士（LLM）将使用两个不同的模板进行提问，以评估答案与检索到的语境之间的关联性。每个模板的评分分别为0、1或2。
- 步骤 2：将每个评分除以 2，标准化为 [0,1] 范围内的数值（例如，0 变为 0.0，1 变为 0.5，2 变为 1.0）。如果两个评分均有效，则最终得分将计算为这些标准化值的平均值；如果只有一个有效，则使用该分数。

- 计算示例：
    - 回答： “阿尔伯特·爱因斯坦出生于 1879 年。”
    - 检索上下文：
    - “阿尔伯特·爱因斯坦出生于 1879 年 3 月 14 日。”
    - “阿尔伯特·爱因斯坦出生于德国符腾堡州乌尔姆。”
    
在此示例中，检索到的上下文提供了阿尔伯特·爱因斯坦的出生日期和地点。由于上下文支持该回应的主张（即使日期仅提供部分），因此两个提示都可能将该回应的扎根性评定为2（完全扎根）。将 2 的分数标准化后，结果为1.0（2/2），将两个标准化评分取平均值，最终的“回应扎根性”得分将保持为1。

#### 相似拉格斯指标
- 忠诚度：此指标衡量回复与检索到的上下文在事实上的一致性，确保回复中的每个陈述都得到所提供的信息的支持。忠诚度得分范围为 0 到 1，得分越高，一致性越好。

- 评分标准分数：这是一种通用指标，可根据用户定义的标准评估响应，并可通过使评分标准与要求相一致来评估答案准确性、上下文相关性或响应依据。

#### 指标比较
**忠诚与回应基础**

- LLM 判断：忠诚度要求两次判断以获得详细的索赔细分和判决，而回应依据则使用两个独立的 LLM 判断。
- 令牌使用：忠诚度消耗更多令牌，而响应基础性则更高效地使用令牌。
- 可解释性：忠实度为每个主张提供了透明的推理，而响应基础性提供了原始分数。
- 稳健评估：忠实度结合用户输入进行全面评估，而响应接地性通过双重 LLM 评估确保一致性。

### 代理或工具的使用
代理或工具的使用工作流程可以从多个维度进行评估。以下是一些可用于评估代理或工具在特定任务中表现的指标。

#### 主题坚持
部署在实际应用中的 AI 系统在与用户交互时，通常需要专注于感兴趣的领域，但 LLM 有时可能会忽略这一限制，直接回答一般性问题。主题依从性指标评估的是 AI 在交互过程中停留在预定义领域的能力。这一指标在对话式 AI 系统中尤为重要，因为在对话式 AI 系统中，AI 需要仅针对与预定义领域相关的查询提供帮助。

`TopicAdherenceScore`需要一组预先定义的主题，AI系统预期这些主题将使用`reference_topics`和提供`user_input`。该指标可以计算主题遵循度的精确度、召回率和 F1 分数，定义为

$\text { Precision }=\frac{\mid \text { Queries that are answered and are adheres to any present reference topics}}{\mid \text { Queries that are answered and are adheres to any present reference topics  + | Queries that are answered and do not adheres to any present reference topics} \mid}$

$\text { Recall }=\frac{\mid \text { Queries that are answered and are adheres to any present reference topics }}{\mid \text { Queries that are answered and are adheres to any present reference topics } \mid+ \text { |Queries that were refused and should have been answered| }}$

$\text { F1 Score }=\frac{2 \times \text { Precision } \times \text { Recall }}{\text { Precision }+ \text { Recall }}$


In [ ]:
from ragas.dataset_schema import  SingleTurnSample, MultiTurnSample, EvaluationDataset
from ragas.messages import HumanMessage,AIMessage,ToolMessage,ToolCall
from ragas.metrics import TopicAdherenceScore


sample_input_4 = [
HumanMessage(content="Can you provide me with details about Einstein's theory of relativity?"),
AIMessage(content="Sure, let me retrieve the relevant information for you.", tool_calls=[
    ToolCall(name="document_search", args={"query": "Einstein's theory of relativity"})
]),
ToolMessage(content="Found relevant documents: 1. Relativity: The Special and the General Theory, 2. General Theory of Relativity by A. Einstein."),
AIMessage(content="I found some documents on Einstein's theory of relativity. Which one would you like to know more about: 'Relativity: The Special and the General Theory' or 'General Theory of Relativity by A. Einstein'?"),
HumanMessage(content="Tell me about the 'General Theory of Relativity'."),
AIMessage(content="Got it! Let me fetch more details from 'General Theory of Relativity by A. Einstein'.", tool_calls=[
    ToolCall(name="document_retrieve", args={"document": "General Theory of Relativity by A. Einstein"})
]),
ToolMessage(content="The document discusses how gravity affects the fabric of spacetime, describing the relationship between mass and spacetime curvature."),
AIMessage(content="The 'General Theory of Relativity' explains how gravity affects the fabric of spacetime and the relationship between mass and spacetime curvature. Would you like more details or a specific explanation?"),
HumanMessage(content="No, that's perfect. By the way, do you know any good recipes for a chocolate cake?"),
AIMessage(content="Sure! Let me find a simple and delicious recipe for a chocolate cake.", tool_calls=[
    ToolCall(name="recipe_search", args={"query": "chocolate cake recipe"})
]),
ToolMessage(content="Here’s a popular recipe for a chocolate cake: Ingredients include flour, sugar, cocoa powder, eggs, milk, and butter. Instructions: Mix dry ingredients, add wet ingredients, and bake at 350°F for 30-35 minutes."),
AIMessage(content="I found a great recipe for chocolate cake! Would you like the full details, or is that summary enough?")
]


sample = MultiTurnSample(user_input=sample_input_4, reference_topics=["science"])
scorer = TopicAdherenceScore(llm = evaluator_llm, mode="precision")
await scorer.multi_turn_ascore(sample)

输出

0.6666666666444444
要更改调用模式，请将mode参数设置为recall。


`scorer = TopicAdherenceScore(llm = evaluator_llm, mode="recall")`

输出

0.99999999995

#### 工具调用准确度

`ToolCallAccuracy`是一个指标，可用于评估 LLM 在识别和调用完成给定任务所需工具方面的性能。该指标需要`user_input`和`reference_tool_calls`来评估 LLM 在识别和调用完成给定任务所需工具方面的性能。该指标通过比较`reference_tool_calls`与 AI 进行的工具调用次数来计算。值的范围在 0 到 1 之间，值越高，性能越好。

In [ ]:
from ragas.metrics import ToolCallAccuracy
from ragas.dataset_schema import  MultiTurnSample
from ragas.messages import HumanMessage,AIMessage,ToolMessage,ToolCall

sample = [
    HumanMessage(content="What's the weather like in New York right now?"),
    AIMessage(content="The current temperature in New York is 75°F and it's partly cloudy.", tool_calls=[
        ToolCall(name="weather_check", args={"location": "New York"})
    ]),
    HumanMessage(content="Can you translate that to Celsius?"),
    AIMessage(content="Let me convert that to Celsius for you.", tool_calls=[
        ToolCall(name="temperature_conversion", args={"temperature_fahrenheit": 75})
    ]),
    ToolMessage(content="75°F is approximately 23.9°C."),
    AIMessage(content="75°F is approximately 23.9°C.")
]

sample = MultiTurnSample(
    user_input=sample,
    reference_tool_calls=[
        ToolCall(name="weather_check", args={"location": "New York"}),
        ToolCall(name="temperature_conversion", args={"temperature_fahrenheit": 75})
    ]
)

scorer = ToolCallAccuracy()
await scorer.multi_turn_ascore(sample)

输出

1.0


中指定的工具调用顺序`reference_tool_calls`将作为理想结果。如果 AI 的工具调用与 的顺序或序列不匹配`reference_tool_calls`，则该指标将返回 0 分。这有助于确保 AI 能够识别并按正确的顺序调用所需工具来完成给定任务。

默认情况下，工具名称和参数会使用精确字符串匹配进行比较。但有时这可能并非最佳选择，例如，如果参数是自然语言字符串。您也可以使用任何 ragas 指标（介于 0 和 1 之间的值）作为距离度量，以识别检索到的上下文是否相关。例如，

In [ ]:
from ragas.metrics._string import NonLLMStringSimilarity
from ragas.metrics._tool_call_accuracy import ToolCallAccuracy

metric = ToolCallAccuracy()
metric.arg_comparison_metric = NonLLMStringSimilarity()

#### 代理目标准确率
代理目标准确率是一个指标，可用于评估 LLM 在识别和实现用户目标方面的表现。这是一个二进制指标，1 表示 AI 已实现目标，0 表示 AI 未实现目标。

##### 参考

`AgentGoalAccuracyWithReference`参考需求进行计算`user_input`，并`reference`评估LLM在识别和实现用户目标方面的表现。注释`reference`将作为理想结果。该指标通过reference与工作流程结束时实现的目标进行比较来计算。

In [ ]:
from ragas.dataset_schema import  MultiTurnSample
from ragas.messages import HumanMessage,AIMessage,ToolMessage,ToolCall
from ragas.metrics import AgentGoalAccuracyWithReference


sample = MultiTurnSample(user_input=[
    HumanMessage(content="Hey, book a table at the nearest best Chinese restaurant for 8:00pm"),
    AIMessage(content="Sure, let me find the best options for you.", tool_calls=[
        ToolCall(name="restaurant_search", args={"cuisine": "Chinese", "time": "8:00pm"})
    ]),
    ToolMessage(content="Found a few options: 1. Golden Dragon, 2. Jade Palace"),
    AIMessage(content="I found some great options: Golden Dragon and Jade Palace. Which one would you prefer?"),
    HumanMessage(content="Let's go with Golden Dragon."),
    AIMessage(content="Great choice! I'll book a table for 8:00pm at Golden Dragon.", tool_calls=[
        ToolCall(name="restaurant_book", args={"name": "Golden Dragon", "time": "8:00pm"})
    ]),
    ToolMessage(content="Table booked at Golden Dragon for 8:00pm."),
    AIMessage(content="Your table at Golden Dragon is booked for 8:00pm. Enjoy your meal!"),
    HumanMessage(content="thanks"),
],
    reference="Table booked at one of the chinese restaurants at 8 pm")

scorer = AgentGoalAccuracyWithReference(llm = evaluator_llm)
await scorer.multi_turn_ascore(sample)

输出

1.0

##### 无参考
`AgentGoalAccuracyWithoutReferenc`e在无参考模式下，该指标将评估 LLM 在无任何参考的情况下识别和实现用户目标的性能。此时，期望结果是根据工作流中的人机交互推断出来的。

In [ ]:
from ragas.dataset_schema import  MultiTurnSample
from ragas.messages import HumanMessage,AIMessage,ToolMessage,ToolCall
from ragas.metrics import AgentGoalAccuracyWithoutReference


sample = MultiTurnSample(user_input=[
    HumanMessage(content="Hey, book a table at the nearest best Chinese restaurant for 8:00pm"),
    AIMessage(content="Sure, let me find the best options for you.", tool_calls=[
        ToolCall(name="restaurant_search", args={"cuisine": "Chinese", "time": "8:00pm"})
    ]),
    ToolMessage(content="Found a few options: 1. Golden Dragon, 2. Jade Palace"),
    AIMessage(content="I found some great options: Golden Dragon and Jade Palace. Which one would you prefer?"),
    HumanMessage(content="Let's go with Golden Dragon."),
    AIMessage(content="Great choice! I'll book a table for 8:00pm at Golden Dragon.", tool_calls=[
        ToolCall(name="restaurant_book", args={"name": "Golden Dragon", "time": "8:00pm"})
    ]),
    ToolMessage(content="Table booked at Golden Dragon for 8:00pm."),
    AIMessage(content="Your table at Golden Dragon is booked for 8:00pm. Enjoy your meal!"),
    HumanMessage(content="thanks"),
])

scorer = AgentGoalAccuracyWithoutReference(llm = evaluator_llm)
await scorer.multi_turn_ascore(sample)

输出

1.0

### 自然语言比较

#### 事实正确性
**事实正确性**
FactualCorrectnessresponse是一种指标，用于比较和评估生成的与的事实准确性reference。此指标用于确定生成的响应与参考的一致程度。事实正确性得分范围从 0 到 1，值越高表示性能越好。为了衡量响应与参考之间的一致性，该指标使用 LLM 首先将响应和参考分解为声明，然后使用自然语言推理来确定响应与参考之间的事实重叠。事实重叠使用精度、召回率和 F1 分数来量化，可以使用mode参数进行控制。

计算真阳性（TP）、假阳性（FP）、假阴性（FN）的公式如下：

True Positive $(\mathrm{TP})=$ Number of claims in response that are present in reference

False Positive $(\mathrm{FP})=$ Number of claims in response that are not present in reference

False Negative $(\mathrm{FN})=$ Number of claims in reference that are not present in response

计算精度、召回率和F1分数的公式如下：

$\begin{gathered}
\text { Precision }=\frac{T P}{(T P+F P)} \\
\text { Recall }=\frac{T P}{(T P+F N)} \\
\text { F1 Score }=\frac{2 \times \text { Precision } \times \text { Recall }}{(\text { Precision }+ \text { Recall })}
\end{gathered}$

In [1]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._factual_correctness import FactualCorrectness


sample = SingleTurnSample(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. I has a height of 1000ft."
)

scorer = FactualCorrectness(llm = evaluator_llm)
await scorer.single_turn_ascore(sample)

/opt/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'evaluator_llm' is not defined

输出

0.67
默认情况下，模式设置为F1，您可以通过设置参数将模式mode更改为precision或recall。


`scorer = FactualCorrectness(llm = evaluator_llm, mode="precision")`
输出

1.0

##### 控制索赔数量
答复和参考文献中的每个句子都可以分解为一个或多个权利要求。单个句子生成的权利要求数量取决于您的申请级别atomicity和coverage要求。

例子

`scorer = FactualCorrectness(mode="precision",atomicity="low")`

输出

1.0


理解原子性和覆盖率
在索赔分解中，有两个重要参数影响输出：
1．原子性
2．覆盖范围
这些参数有助于控制生成的索赔的粒度和完整性。
原子性
原子性是指将一个句子分解成最小、有意义的组成部分的程度。您可以根据自己需要高度详细的声明还是更综合的视图进行调整。
－高原子性：句子被分解成基本且不可分割的声明。这会产生多个更小的声明，每个声明代表一条不同的信息。

示例：－原句：－＂阿尔伯特•爱因斯坦是一位德国理论物理学家，他发展了相对论并对量子力学做出了贡献。＂－分解后的主张：－＂阿尔伯特•爱因斯坦是一位德国理论物理学家。＂－＂阿尔伯特•爱因斯坦发展了相对论。＂－＂阿尔伯特•爱因斯坦对量子力学做出了贡献。＂
－低原子性：句子保持更完整，从而导致可能包含多条信息的声明更少。
示例：－原句：－＂阿尔伯特•爱因斯坦是一位德国理论物理学家，他发展了相对论，并对量子力学做出了贡献。＂－分解后的主张：－＂阿尔伯特•爱因斯坦是一位德国理论物理学家，他发展了相对论，并对量子力学做出了贡献。＂

覆盖范围
覆盖度是指主张对原句信息的全面性。它可以调整为涵盖所有细节，也可以概括内容。
－高覆盖率：分解后的声明捕获了原始句子中存在的所有信息，保留了每个细节。
示例：－原句：－＂玛丽•居里是一位波兰裔法国物理学家和化学家，她对放射性进行了开创性的研究。＂－分解后的声明：－＂玛丽•居里是一位波兰物理学家。＂－＂玛丽•居里是一位归化法国物理学家。＂
- ＂玛丽•居里是一位化学家。＂－＂玛丽•居里对放射性进行了开创性的研究。＂
- 覆盖范围低：分解后的权利要求仅涵盖要点，省略一些细节以提供更为概括的视图。

示例：－原句：－＂玛丽•居里是一位波兰裔法国物理学家和化学家，她对放射性进行了开创性的研究。＂－分解后的主张：－＂玛丽•居里是一位物理学家。＂－＂玛丽•居里对放射性进行了研究。＂

结合原子性和覆盖率
通过调整原子性和覆盖率，您可以自定义细节和完整性级别以满足特定用例的需求。
－高原子性和高覆盖率：产生涵盖原始句子所有方面的高度详细和全面的声明。
例如：－原句：－＂查尔斯•巴贝奇是一位英国数学家、哲学家、发明家和机械工程师。＂－分解后的声明：－＂查尔斯•巴贝奇是一位英国数学家。＂－＂查尔斯•巴贝奇是一位哲学家。＂－＂查尔斯•巴贝奇是
- 位发明家。＂－＂查尔斯•巴贝奇是一位机械工程师。＂
- 低原子性和低覆盖率：提出较少的主张，较少的细节，总结主要思想而不涉及细节。

示例：－原句：－＂查尔斯•巴贝奇是一位英国数学家、哲学家、发明家和机械工程师。＂－分解后的主张：－＂查尔斯•巴贝奇是一位英国数学家。＂－＂查尔斯•巴贝奇是一位发明家。＂

实际应用
- 当您需要详细全面的细分以进行深入分析或信息提取时，请使用高原子性和高覆盖率。
- 当只需要关键信息（例如摘要）时，使用低原子性和低覆盖率。

这种控制索赔数量的灵活性有助于确保信息以适合您的应用程序要求的粒度级别呈现。

#### 语义相似性
答案语义相似度的概念是指评估生成的答案与基本事实之间的语义相似度。该评估基于 和ground truth，answer值在 0 到 1 的范围内。分数越高，表示生成的答案与基本事实之间的匹配程度越高。

测量答案之间的语义相似度可以为生成响应的质量提供有价值的见解。本次评估采用双编码器模型来计算语义相似度得分。



In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import SemanticSimilarity
from ragas.embeddings import LangchainEmbeddingsWrapper

sample = SingleTurnSample(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. It has a height of 1000ft."
)

scorer = SemanticSimilarity(embeddings=LangchainEmbeddingsWrapper(evaluator_embedding))
await scorer.single_turn_ascore(sample)

输出

0.8151371879226978


参考：阿尔伯特·爱因斯坦的相对论彻底改变了我们对宇宙的理解。”

高相似度答案：爱因斯坦的突破性相对论改变了我们对宇宙的理解。

相似度低的答案：艾萨克·牛顿的运动定律极大地影响了古典物理学。

让我们来看看第一个答案的答案相似度是如何计算的：

- 步骤 1：使用指定的嵌入模型对基本事实答案进行矢量化。
- 第 2 步：使用相同的嵌入模型对生成的答案进行矢量化。
- 步骤3：计算两个向量之间的余弦相似度。

#### 传统的 NLP 指标
##### 非 LLM 字符串相似性
NonLLMStringSimilarity指标使用传统的字符串距离度量（例如 Levenshtein、Hamming 和 Jaro）来衡量参考文本与响应之间的相似度。此指标有助于评估文本与response参考reference文本的相似度，而无需依赖大型语言模型 (LLM)。该指标返回 0 到 1 之间的分数，其中 1 表示响应与参考文本完全匹配。这是一个非基于 LLM 的指标。



In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._string import NonLLMStringSimilarity

sample = SingleTurnSample(
    response="The Eiffel Tower is located in India.",
    reference="The Eiffel Tower is located in Paris."
)

scorer = NonLLMStringSimilarity()
await scorer.single_turn_ascore(sample)

可以从 DistanceMeasure中选择可用的字符串距离度量。
以下是使用汉明距离的示例。

In [ ]:
from ragas.metrics._string import NonLLMStringSimilarity, DistanceMeasure

scorer = NonLLMStringSimilarity(distance_measure=DistanceMeasure.HAMMING)

##### BLEU 分数

分数BleuScore是一种用于评估质量的指标，response通过将其与进行比较来衡量reference。它基于 n-gram 精度和简洁性惩罚来衡量响应与参考之间的相似性。BLEU 分数最初用于评估机器翻译系统，但也用于其他自然语言处理任务。BLEU 分数的范围是 0 到 1，其中 1 表示响应与参考完全匹配。这是一个非基于 LLM 的指标。

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import BleuScore

sample = SingleTurnSample(
    response="The Eiffel Tower is located in India.",
    reference="The Eiffel Tower is located in Paris."
)

scorer = BleuScore()
await scorer.single_turn_ascore(sample)

输出

0.7071067811865478

##### ROUGE 分数
分数RougeScore是一组用于评估自然语言生成质量的指标。它基于 n-gram 召回率、准确率和 F1 分数来衡量生成结果response与reference文本之间的重叠度。ROUGE 分数的范围是 0 到 1，其中 1 表示响应与参考文献完全匹配。这是一个非基于 LLM 的指标。





In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import RougeScore

sample = SingleTurnSample(
    response="The Eiffel Tower is located in India.",
    reference="The Eiffel Tower is located in Paris."
)

scorer = RougeScore()
await scorer.single_turn_ascore(sample)

输出

0.8571428571428571

您可以更改rouge_type为rouge1或rougeL分别基于单元或最长公共子序列计算 ROUGE 分数。

In [ ]:
scorer = RougeScore(rouge_type="rouge1")

您可以将模式更改为 precision、recall 或 fmeasure，以分别根据 precision、recall 或 F1 分数计算 ROUGE 分数

In [ ]:
scorer = RougeScore(mode="recall")

##### 精确匹配
该ExactMatch指标用于检查响应是否与参考文本完全相同。在需要确保生成的响应与预期输出逐字匹配的情况下，该指标非常有用。例如，工具调用中的参数等。如果响应与参考文本完全匹配，则该指标返回 1，否则返回 0。

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import ExactMatch

sample = SingleTurnSample(
    response="India",
    reference="Paris"
)

scorer = ExactMatch()
await scorer.single_turn_ascore(sample)

##### 字符串存在
该StringPresence指标检查响应是否包含参考文本。在需要确保生成的响应包含特定关键字或短语的情况下，该指标非常有用。如果响应包含参考文本，则该指标返回 1，否则返回 0。


In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import StringPresence

sample = SingleTurnSample(
    response="The Eiffel Tower is located in India.",
    reference="Eiffel Tower"
)
scorer = StringPresence()
await scorer.single_turn_ascore(sample)